# Zadanie 1 — Toy Example filtru 1D

Lokalizacja robota filtrem cząsteczkowym na cyklicznym świecie (jak w `ZRI_c1.pdf`).

## Ustawienia świata

```text
świat = [ ściana, drzwi, ściana, ściana, drzwi ]
```

- Robot: ruch w prawo `+1 (mod 5)`
- Sensor: rozróżnia drzwi i ścianę z dokładnością `0.8`

In [208]:
import numpy as np
import matplotlib.pyplot as plt

sciana = 0
drzwi = 1

swiat = np.array([sciana, drzwi, sciana, sciana, drzwi])
nazwy = {sciana: "ściana", drzwi: "drzwi"}

p_poprawny = 0.8
p_bledny = 0.2

print("świat =", [nazwy[x] for x in swiat])
print("indeksy:", list(range(len(swiat))))

świat = ['ściana', 'drzwi', 'ściana', 'ściana', 'drzwi']
indeksy: [0, 1, 2, 3, 4]


---
## Krok 1 — Inicjalizacja

Generujemy 10 cząsteczek w sposób równomierny:
`[0, 1, 2, 3, 4, 0, 2, 3, 4, 1]`

In [59]:
czasteczki = np.array([0, 1, 2, 3, 4, 0, 2, 3, 4, 1])
print("cząsteczki =", czasteczki.tolist())
print("typy pól  =", [nazwy[swiat[i]] for i in czasteczki])

cząsteczki = [0, 1, 2, 3, 4, 0, 2, 3, 4, 1]
typy pól  = ['ściana', 'drzwi', 'ściana', 'ściana', 'drzwi', 'ściana', 'ściana', 'ściana', 'drzwi', 'drzwi']


---
## Krok 2 — Pierwszy pomiar: drzwi

Wagi:
- `0.8` jeśli pole to drzwi
- `0.2` w przeciwnym razie

In [221]:
czasteczki = np.array([0, 1, 2, 3, 4, 0, 2, 3, 4, 1])  # Krok 1

pomiar = drzwi

w = []
for pozycja in czasteczki:
    if swiat[pozycja] == pomiar:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)

S = w.sum()
w_prim = np.round(w / S, 3)  

print("2.a")
print("S =", round(S, 2))
print("\n")

print("2.b")
print("w  (przed normalizacją) =", np.round(w, 3).tolist())

print("w' (po normalizacji)    =", w_prim.tolist())
print(f"suma w'= {round(w_prim.sum(), 3)} ≈ 1")

2.a
S = 4.4


2.b
w  (przed normalizacją) = [0.2, 0.8, 0.2, 0.2, 0.8, 0.2, 0.2, 0.2, 0.8, 0.8]
w' (po normalizacji)    = [0.045, 0.182, 0.045, 0.045, 0.182, 0.045, 0.045, 0.045, 0.182, 0.182]
suma w'= 0.998 ≈ 1


### Resampling (ruletka, u i CDF)

Losowe wartości z PDF:
`u = [0.05, 0.18, 0.32, 0.41, 0.55, 0.62, 0.71, 0.86, 0.93, 0.99]`

Dla każdego `u_j` wybieramy **najmniejszy indeks i**, dla którego `CDF_i ≥ u_j`.

In [101]:
def zbuduj_CDF(wagi):
    return np.cumsum(wagi)


def resampling_ruletka(czasteczki, wagi, u):
    """Metoda ruletkowa jak w PDF: CDF + wartości u."""
    CDF = zbuduj_CDF(wagi)
    indeksy = []
    for uj in u:
        i = int(np.searchsorted(CDF, uj, side="left"))
        i = min(i, len(czasteczki) - 1)
        indeksy.append(i)
    indeksy = np.array(indeksy)
    nowe_czasteczki = czasteczki[indeksy].copy()
    return nowe_czasteczki, indeksy, CDF


# zawsze od listy z Kroku 1 (żeby CDF było jak w PDF przy ponownym Run)
czasteczki = np.array([0, 1, 2, 3, 4, 0, 2, 3, 4, 1])
w = []
for pozycja in czasteczki:
    if swiat[pozycja] == drzwi:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)
w_prim = np.round(w / w.sum(), 3)

u = np.array([0.05, 0.18, 0.32, 0.41, 0.55, 0.62, 0.71, 0.86, 0.93, 0.99])
czasteczki, indeksy, CDF = resampling_ruletka(czasteczki, w_prim, u)

print("2.c Resampling (ruletka/CDF)")
print("w'  =", w_prim.tolist())
print("CDF ≈", np.round(CDF, 3).tolist())
print("wybrane indeksy i =", indeksy.tolist())
print("nowe cząsteczki   =", czasteczki.tolist())

# Tabela 1 jak w PDF
from IPython.display import Markdown, display

linie = [
    "**Tabela 1:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)",
    "",
    "| $u_j$ | Przedział CDF | Wybrany indeks $i$ |",
    "|------:|:--------------|-------------------:|",
]
for uj, i in zip(u, indeksy):
    prawy = float(CDF[i])
    if i == 0:
        przedzial = f"u ≤ {prawy:.3f}"
    else:
        lewy = float(CDF[i - 1])
        przedzial = f"{lewy:.3f} < u ≤ {prawy:.3f}"
    linie.append(f"| {uj} | {przedzial} | {int(i)} |")

display(Markdown("\n".join(linie)))

2.c Resampling (ruletka/CDF)
w'  = [0.045, 0.182, 0.045, 0.045, 0.182, 0.045, 0.045, 0.045, 0.182, 0.182]
CDF ≈ [0.045, 0.227, 0.272, 0.317, 0.499, 0.544, 0.589, 0.634, 0.816, 0.998]
wybrane indeksy i = [1, 1, 4, 4, 6, 7, 8, 9, 9, 9]
nowe cząsteczki   = [1, 1, 4, 4, 2, 3, 4, 1, 1, 1]


**Tabela 1:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)

| $u_j$ | Przedział CDF | Wybrany indeks $i$ |
|------:|:--------------|-------------------:|
| 0.05 | 0.045 < u ≤ 0.227 | 1 |
| 0.18 | 0.045 < u ≤ 0.227 | 1 |
| 0.32 | 0.317 < u ≤ 0.499 | 4 |
| 0.41 | 0.317 < u ≤ 0.499 | 4 |
| 0.55 | 0.544 < u ≤ 0.589 | 6 |
| 0.62 | 0.589 < u ≤ 0.634 | 7 |
| 0.71 | 0.634 < u ≤ 0.816 | 8 |
| 0.86 | 0.816 < u ≤ 0.998 | 9 |
| 0.93 | 0.816 < u ≤ 0.998 | 9 |
| 0.99 | 0.816 < u ≤ 0.998 | 9 |

| | |
|--|--|
| wybrane i | `[1, 1, 4, 4, 6, 7, 8, 9, 9, 9]` |
| indeksy | `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` |
| pozycja na świecie | `[0, 1, 2, 3, 4, 0, 2, 3, 4, 1]` |
| nowe cząsteczki | `[1, 1, 4, 4, 2, 3, 4, 1, 1, 1]` |

---
## Krok 3 — Pierwszy ruch: +1 w prawo

`cząsteczki = (cząsteczki + 1) mod 5`

In [216]:
def move(czasteczki, rozmiar_swiata, krok=1):
    return (czasteczki + krok) % rozmiar_swiata


def rozklad(czasteczki, rozmiar_swiata):
    counts = np.bincount(czasteczki, minlength=rozmiar_swiata)
    return {i: int(c) for i, c in enumerate(counts) if c > 0}


czasteczki = np.array([1, 1, 4, 4, 2, 3, 4, 1, 1, 1])  # po kroku 2.c

print("przed ruchem:", czasteczki.tolist())
czasteczki = move(czasteczki, len(swiat))
print("po ruchu +1: ", czasteczki.tolist())
print("rozkład:     ", rozklad(czasteczki, len(swiat)))

przed ruchem: [1, 1, 4, 4, 2, 3, 4, 1, 1, 1]
po ruchu +1:  [2, 2, 0, 0, 3, 4, 0, 2, 2, 2]
rozkład:      {0: 3, 2: 5, 3: 1, 4: 1}


---
## Krok 4 — Drugi pomiar: ściana

Ściana na pozycjach `{0, 2, 3}`.

Wagi:
- `0.8` jeśli pole to ściana
- `0.2` w przeciwnym razie

In [222]:
czasteczki = np.array([2, 2, 0, 0, 3, 4, 0, 2, 2, 2])  # po kroku 3

pomiar = sciana

w = []
for pozycja in czasteczki:
    if swiat[pozycja] == pomiar:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)

S = w.sum()
w_prim = np.round(w / S, 3)

print("4.a")
print("S =", round(S, 2))
print("\n")

print("4.b")
print("w  (przed normalizacją) =", np.round(w, 3).tolist())

print("w' (po normalizacji)    =", w_prim.tolist())
print(f"suma w'= {round(w_prim.sum(), 3)} ≈ 1")

4.a
S = 7.4


4.b
w  (przed normalizacją) = [0.8, 0.8, 0.8, 0.8, 0.8, 0.2, 0.8, 0.8, 0.8, 0.8]
w' (po normalizacji)    = [0.108, 0.108, 0.108, 0.108, 0.108, 0.027, 0.108, 0.108, 0.108, 0.108]
suma w'= 0.999 ≈ 1


### Resampling (ruletka, u i CDF)

Losowe wartości z PDF:
`u = [0.05, 0.12, 0.19, 0.24, 0.41, 0.43, 0.74, 0.88, 0.93, 0.97]`

Dla każdego `u_j` wybieramy **najmniejszy indeks i**, dla którego `CDF_i ≥ u_j`.

In [223]:
# zawsze od listy po kroku 3
czasteczki = np.array([2, 2, 0, 0, 3, 4, 0, 2, 2, 2])
w = []
for pozycja in czasteczki:
    if swiat[pozycja] == sciana:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)
w_prim = np.round(w / w.sum(), 3)

u = np.array([0.05, 0.12, 0.19, 0.24, 0.41, 0.43, 0.74, 0.88, 0.93, 0.97])
czasteczki, indeksy, CDF = resampling_ruletka(czasteczki, w_prim, u)

print("4.c Resampling (ruletka/CDF)")
print("w'  =", w_prim.tolist())
print("CDF ≈", np.round(CDF, 3).tolist())
print("wybrane indeksy i =", indeksy.tolist())
print("nowe cząsteczki   =", czasteczki.tolist())

from IPython.display import Markdown, display

linie = [
    "**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)",
    "",
    "| $u_j$ | Przedział CDF | Wybrany indeks $i$ |",
    "|------:|:--------------|-------------------:|",
]
for uj, i in zip(u, indeksy):
    prawy = float(CDF[i])
    if i == 0:
        przedzial = f"u ≤ {prawy:.3f}"
    else:
        lewy = float(CDF[i - 1])
        przedzial = f"{lewy:.3f} < u ≤ {prawy:.3f}"
    linie.append(f"| {uj} | {przedzial} | {int(i)} |")

display(Markdown("\n".join(linie)))

4.c Resampling (ruletka/CDF)
w'  = [0.108, 0.108, 0.108, 0.108, 0.108, 0.027, 0.108, 0.108, 0.108, 0.108]
CDF ≈ [0.108, 0.216, 0.324, 0.432, 0.54, 0.567, 0.675, 0.783, 0.891, 0.999]
wybrane indeksy i = [0, 1, 1, 2, 3, 3, 7, 8, 9, 9]
nowe cząsteczki   = [2, 2, 2, 0, 0, 0, 2, 2, 2, 2]


**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)

| $u_j$ | Przedział CDF | Wybrany indeks $i$ |
|------:|:--------------|-------------------:|
| 0.05 | u ≤ 0.108 | 0 |
| 0.12 | 0.108 < u ≤ 0.216 | 1 |
| 0.19 | 0.108 < u ≤ 0.216 | 1 |
| 0.24 | 0.216 < u ≤ 0.324 | 2 |
| 0.41 | 0.324 < u ≤ 0.432 | 3 |
| 0.43 | 0.324 < u ≤ 0.432 | 3 |
| 0.74 | 0.675 < u ≤ 0.783 | 7 |
| 0.88 | 0.783 < u ≤ 0.891 | 8 |
| 0.93 | 0.891 < u ≤ 0.999 | 9 |
| 0.97 | 0.891 < u ≤ 0.999 | 9 |

**Wynik resamplingu (krok 4):**

| | |
|--|--|
| wybrane i | `[0, 1, 1, 2, 3, 3, 7, 8, 9, 9]` |
| indeksy | `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` |
| pozycja na świecie | `[2, 2, 0, 0, 3, 4, 0, 2, 2, 2]` |
| nowe cząsteczki | `[2, 2, 2, 0, 0, 0, 2, 2, 2, 2]` |

---
## Krok 5 — Drugi ruch: +1 w prawo

`cząsteczki = (cząsteczki + 1) mod 5`

In [224]:
czasteczki = np.array([2, 2, 2, 0, 0, 0, 2, 2, 2, 2]) 

print("przed ruchem:", czasteczki.tolist())
czasteczki = move(czasteczki, len(swiat))
print("po ruchu +1: ", czasteczki.tolist())
print("rozkład:     ", rozklad(czasteczki, len(swiat)))

przed ruchem: [2, 2, 2, 0, 0, 0, 2, 2, 2, 2]
po ruchu +1:  [3, 3, 3, 1, 1, 1, 3, 3, 3, 3]
rozkład:      {1: 3, 3: 7}


---
## Krok 6 — Trzeci pomiar: ściana

Ściana na pozycjach `{0, 2, 3}`.

Wagi:
- `0.8` jeśli pole to ściana
- `0.2` w przeciwnym razie

In [226]:
czasteczki = np.array([3, 3, 3, 1, 1, 1, 3, 3, 3, 3])  # po kroku 5

pomiar = sciana

w = []
for pozycja in czasteczki:
    if swiat[pozycja] == pomiar:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)

S = w.sum()
w_prim = np.round(w / S, 5)

print("6.a")
print("S =", round(S, 2))
print("\n")

print("6.b")
print("w  (przed normalizacją) =", np.round(w, 3).tolist())

print("w' (po normalizacji)    =", w_prim.tolist())
print("suma w' ≈", round(w_prim.sum(), 3))

6.a
S = 6.2


6.b
w  (przed normalizacją) = [0.8, 0.8, 0.8, 0.2, 0.2, 0.2, 0.8, 0.8, 0.8, 0.8]
w' (po normalizacji)    = [0.12903, 0.12903, 0.12903, 0.03226, 0.03226, 0.03226, 0.12903, 0.12903, 0.12903, 0.12903]
suma w' ≈ 1.0


### Resampling (ruletka, u i CDF)

Losowe wartości z PDF:
`u = [0.05, 0.12, 0.20, 0.33, 0.45, 0.60, 0.72, 0.81, 0.93, 0.99]`

In [227]:
# zawsze od listy po kroku 5
czasteczki = np.array([3, 3, 3, 1, 1, 1, 3, 3, 3, 3])
w = []
for pozycja in czasteczki:
    if swiat[pozycja] == sciana:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)
w_prim = np.round(w / w.sum(), 5)

u = np.array([0.05, 0.12, 0.20, 0.33, 0.45, 0.60, 0.72, 0.81, 0.93, 0.99])
czasteczki, indeksy, CDF = resampling_ruletka(czasteczki, w_prim, u)

print("6.c Resampling (ruletka/CDF)")
print("w'  =", w_prim.tolist())
print("CDF ≈", np.round(CDF, 4).tolist())
print("wybrane indeksy i =", indeksy.tolist())
print("nowe cząsteczki   =", czasteczki.tolist())

from IPython.display import Markdown, display

linie = [
    "**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)",
    "",
    "| $u_j$ | Przedział CDF | Wybrany indeks $i$ |",
    "|------:|:--------------|-------------------:|",
]
for uj, i in zip(u, indeksy):
    prawy = float(CDF[i])
    if i == 0:
        przedzial = f"u ≤ {prawy:.3f}"
    else:
        lewy = float(CDF[i - 1])
        przedzial = f"{lewy:.4f} < u ≤ {prawy:.4f}"
    linie.append(f"| {uj} | {przedzial} | {int(i)} |")

display(Markdown("\n".join(linie)))

6.c Resampling (ruletka/CDF)
w'  = [0.12903, 0.12903, 0.12903, 0.03226, 0.03226, 0.03226, 0.12903, 0.12903, 0.12903, 0.12903]
CDF ≈ [0.129, 0.2581, 0.3871, 0.4194, 0.4516, 0.4839, 0.6129, 0.7419, 0.871, 1.0]
wybrane indeksy i = [0, 0, 1, 2, 4, 6, 7, 8, 9, 9]
nowe cząsteczki   = [3, 3, 3, 3, 1, 3, 3, 3, 3, 3]


**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)

| $u_j$ | Przedział CDF | Wybrany indeks $i$ |
|------:|:--------------|-------------------:|
| 0.05 | u ≤ 0.129 | 0 |
| 0.12 | u ≤ 0.129 | 0 |
| 0.2 | 0.1290 < u ≤ 0.2581 | 1 |
| 0.33 | 0.2581 < u ≤ 0.3871 | 2 |
| 0.45 | 0.4194 < u ≤ 0.4516 | 4 |
| 0.6 | 0.4839 < u ≤ 0.6129 | 6 |
| 0.72 | 0.6129 < u ≤ 0.7419 | 7 |
| 0.81 | 0.7419 < u ≤ 0.8710 | 8 |
| 0.93 | 0.8710 < u ≤ 1.0000 | 9 |
| 0.99 | 0.8710 < u ≤ 1.0000 | 9 |

**Wynik resamplingu (krok 6):**

| | |
|--|--|
| wybrane i | `[0, 0, 1, 2, 4, 6, 7, 8, 9, 9]` |
| indeksy | `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` |
| pozycja na świecie | `[3, 3, 3, 1, 1, 1, 3, 3, 3, 3]` |
| nowe cząsteczki | `[3, 3, 3, 3, 1, 3, 3, 3, 3, 3]` |

### Wniosek

Masa rozkładu koncentruje się na polu **3** (ściana).

Po dwóch ruchach `+1`:

$$3 \xleftarrow{-1} 2 \xleftarrow{-1} 1$$

Pozycja startowa to **drzwi pod indeksem 1**. Pomiary: drzwi → ściana → ściana.